In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBRFClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import pandas as pd

In [ ]:
df = pd.read_csv('../data/processed/train.csv')

X = df.drop(columns=['Survived'])
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,IsAlone,Embarked_Q,Embarked_S
0,3,0,-0.565736,1,0,-0.502445,2,0,False,True
1,1,1,0.663861,1,0,0.786845,2,0,False,False
2,3,1,-0.258337,0,0,-0.488854,1,1,False,True
3,1,1,0.433312,1,0,0.420730,2,0,False,True
4,3,0,0.433312,0,0,-0.486337,1,1,False,True


In [65]:
cv = StratifiedKFold(
    n_splits=5, 
    shuffle=True, 
    random_state=42
    )

In [66]:
models = {
    'LogisticRegression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'RandomForest': RandomForestClassifier(),
    'XGBoostRF': XGBRFClassifier(),
    'CatBoost': CatBoostClassifier(),
    'LightGBM': LGBMClassifier()
}


c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\xgboost\core.py:569: FutureWarning: `XGBRFClassifier` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.
  return func(**kwargs)


In [67]:
results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='accuracy'
    )
    results.append({
        'model': name,
        'accuracy': scores.mean(),
        'std': scores.std()
    })

c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\xgboost\core.py:569: FutureWarning: `XGBRFClassifier` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.
  return func(**kwargs)
c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\xgboost\core.py:569: FutureWarning: `XGBRFClassifier` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest imple

Learning rate set to 0.008911
0:	learn: 0.6871164	total: 2.34ms	remaining: 2.34s
1:	learn: 0.6817415	total: 4.16ms	remaining: 2.08s
2:	learn: 0.6774290	total: 5.85ms	remaining: 1.94s
3:	learn: 0.6717174	total: 7.59ms	remaining: 1.89s
4:	learn: 0.6664005	total: 9.34ms	remaining: 1.86s
5:	learn: 0.6608016	total: 11.4ms	remaining: 1.89s
6:	learn: 0.6557676	total: 13.3ms	remaining: 1.88s
7:	learn: 0.6507351	total: 15ms	remaining: 1.86s
8:	learn: 0.6456359	total: 16.7ms	remaining: 1.84s
9:	learn: 0.6408423	total: 18.4ms	remaining: 1.82s
10:	learn: 0.6360542	total: 20.2ms	remaining: 1.81s
11:	learn: 0.6311772	total: 21.9ms	remaining: 1.8s
12:	learn: 0.6268757	total: 23.7ms	remaining: 1.8s
13:	learn: 0.6217249	total: 26.6ms	remaining: 1.87s
14:	learn: 0.6165523	total: 28.5ms	remaining: 1.87s
15:	learn: 0.6123179	total: 30.3ms	remaining: 1.86s
16:	learn: 0.6083493	total: 32ms	remaining: 1.85s
17:	learn: 0.6039497	total: 33.7ms	remaining: 1.84s
18:	learn: 0.6003847	total: 34.9ms	remaining: 1.8s

In [68]:
results_without_params = pd.DataFrame(results)
results_without_params.sort_values('accuracy', ascending=False)

,model,accuracy,std
4,XGBoostRF,0.840594,0.027819
5,CatBoost,0.835007,0.015822
6,LightGBM,0.833884,0.022149
3,RandomForest,0.819277,0.018824
1,KNN,0.805813,0.021091
0,LogisticRegression,0.800251,0.017636
2,DecisionTree,0.774396,0.015026


In [69]:
experiments = {
    'LogisticRegression': (
        LogisticRegression(max_iter=1000),
        {'C': [0.01, 0.1, 1, 10, 100]},
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': [3, 5, 7, 11, 15, 21],
         'weights': ['uniform', 'distance']},
    ),
    'DecisionTree': (
        DecisionTreeClassifier(random_state=42),
        {'max_depth': [3, 4, 5, 7, 10],
         'min_samples_leaf': [1, 5, 10, 20]},
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [200, 400],
         'max_depth': [4, 6, 8, None],
         'min_samples_leaf': [1, 3, 5]},
    ),
    'XGBRF': (
        XGBRFClassifier(random_state=42),
        {'n_estimators': [200, 400],
         'max_depth': [3, 5, 7],
         'subsample': [0.7, 0.9]},
    ),
    'CatBoost': (
        CatBoostClassifier(random_state=42, verbose=0),
        {'depth': [3, 5, 7],
         'learning_rate': [0.03, 0.1],
         'iterations': [200, 400]},
    ),
    'LightGBM': (
        LGBMClassifier(random_state=42, verbose=-1),
        {'num_leaves': [7, 15, 31],
         'learning_rate': [0.03, 0.1],
         'n_estimators': [100, 300],
         'min_child_samples': [10, 20]},
    ),
}

c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\xgboost\core.py:569: FutureWarning: `XGBRFClassifier` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `sklearn.ensemble`.
  return func(**kwargs)


In [70]:
rgsresults = []

for name, (model, grid) in experiments.items():
    rgs = RandomizedSearchCV(model, grid, n_iter=20, cv=cv, 
                            scoring='accuracy', n_jobs=-1, random_state=42)
    
    rgs.fit(X, y)
    best = rgs.cv_results_
    i = rgs.best_index_
    rgsresults.append({
        'model': name,
        'cv_mean': rgs.best_score_,
        'cv_std': best['std_test_score'][i],
        'best_params': rgs.best_params_,
    })

c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\sklearn\model_selection\_search.py:326: UserWarning: The total space of parameters 5 is smaller than n_iter=20. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\sklearn\model_selection\_search.py:326: UserWarning: The total space of parameters 12 is smaller than n_iter=20. Running 12 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\User\miniconda3\envs\mlpractice\Lib\site-packages\xgboost\core.py:569: FutureWarning: `XGBRFClassifier` is deprecated and will be removed in a future release. The estimator is a thin wrapper over the boosting interface and does not implement a conventional random forest; features like early stopping are unsupported. Set `num_parallel_tree` along with `n_estimators=1` on the corresponding boosting estimator instead, or use a dedicated random forest implementation like those in `s

In [71]:
results_with_params = pd.DataFrame(rgsresults)
results_with_params.sort_values('cv_mean', ascending=False)

,model,cv_mean,cv_std,best_params
6,LightGBM,0.842872,0.020721,"{'num_leaves': 7, 'n_estimators': 100, 'min_ch..."
5,CatBoost,0.842847,0.020269,"{'learning_rate': 0.1, 'iterations': 200, 'dep..."
3,RandomForest,0.839495,0.012231,"{'n_estimators': 400, 'min_samples_leaf': 3, '..."
4,XGBRF,0.836106,0.025151,"{'subsample': 0.9, 'n_estimators': 200, 'max_d..."
2,DecisionTree,0.831668,0.013971,"{'min_samples_leaf': 5, 'max_depth': 10}"
1,KNN,0.805813,0.021091,"{'weights': 'uniform', 'n_neighbors': 5}"
0,LogisticRegression,0.802498,0.017168,{'C': 0.1}
